<a href="https://colab.research.google.com/github/ottrindade1963/Analise-industrial-emergentes-Orfeu/blob/main/Extra%C3%A7%C3%A3o%20Emergentes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Instalação da Biblioteca WDATA**

In [15]:
!pip install wbdata

# **Importar bibliotecas**

In [20]:
import pandas as pd
import numpy as np
import wbdata
import requests
import zipfile
import io
import os
from datetime import datetime
import matplotlib.pyplot as plt

print("Bibliotecas carregadas com sucesso!")

Bibliotecas carregadas com sucesso!


# **Obter lista de países em desenvolvimento**

In [41]:
# Criar subpasta data/raw (se não existir)
os.makedirs('data/raw', exist_ok=True)

# Obter classificação de renda dos países (corrigido)
print("Obtendo dados dos países...")
paises_info = wbdata.get_countries()  # 👈 atenção ao plural
df_paises = pd.DataFrame(paises_info)

# Extrair código e nome do grupo de renda
df_paises['income_code'] = df_paises['incomeLevel'].apply(lambda x: x['id'] if isinstance(x, dict) else None)
df_paises['income_name'] = df_paises['incomeLevel'].apply(lambda x: x['value'] if isinstance(x, dict) else None)

# Selecionar colunas relevantes
df_paises = df_paises[['id', 'name', 'income_code', 'income_name', 'region']].copy()
df_paises.rename(columns={'id': 'codigo_pais', 'name': 'nome_pais'}, inplace=True)

# Filtrar países que NÃO são de alta renda (HIC) → países em desenvolvimento
paises_dev = df_paises[df_paises['income_code'] != 'HIC'].copy()
paises_dev_id = paises_dev['codigo_pais'].tolist()

print(f"✅ Total de países em desenvolvimento: {len(paises_dev_id)}")

# Salvar a lista de países em CSV e Excel
paises_dev.to_csv('data/raw/paises_em_desenvolvimento.csv', index=False)
paises_dev.to_excel('data/raw/paises_em_desenvolvimento.xlsx', index=False)

print("✅ Lista de países salva em:")
print("   - data/raw/paises_em_desenvolvimento.csv")
print("   - data/raw/paises_em_desenvolvimento.xlsx")

Obtendo dados dos países...
✅ Total de países em desenvolvimento: 210
✅ Lista de países salva em:
   - data/raw/paises_em_desenvolvimento.csv
   - data/raw/paises_em_desenvolvimento.xlsx


# **Definir indicadores do WDI e baixar dados**

In [45]:
import os
import pandas as pd
import wbdata

os.makedirs('data/raw', exist_ok=True)

print("Obtendo países do Banco Mundial...")
paises_info = wbdata.get_countries()
df_paises = pd.DataFrame(paises_info)

# Extrair informações
df_paises['region_name'] = df_paises['region'].apply(lambda x: x.get('value') if isinstance(x, dict) else None)
df_paises['income_code'] = df_paises['incomeLevel'].apply(lambda x: x.get('id') if isinstance(x, dict) else None)
df_paises.rename(columns={'id': 'codigo_pais', 'name': 'nome_pais'}, inplace=True)

# Filtro correto: apenas países (não agregados) e não HIC
paises_validos = df_paises[
    (df_paises['region_name'].notna()) &
    (df_paises['region_name'] != 'Aggregates') &
    (df_paises['income_code'].notna()) &
    (df_paises['income_code'] != 'HIC') &
    (df_paises['codigo_pais'].str.match(r'^[A-Z]{3}$'))
].copy()

paises_dev_id = paises_validos['codigo_pais'].tolist()
print(f"✅ Países em desenvolvimento (válidos): {len(paises_dev_id)}")

# Salvar lista limpa
paises_validos.to_csv('data/raw/paises_em_desenvolvimento_limpos.csv', index=False)

# Indicadores
indicadores_wdi = {
    'NY.GDP.PCAP.PP.KD': 'pib_per_capita_ppc',
    'NE.GDI.FTOT.ZS': 'formacao_bruta_capital_fixo_percent_pib',
    'SE.SEC.ENRR': 'matricula_ensino_secundario_percent',
    'NE.TRD.GNFS.ZS': 'comercio_percent_pib',
    'BX.KLT.DINV.WD.GD.ZS': 'investimento_estrangeiro_direto_percent_pib',
    'SP.POP.TOTL': 'populacao_total',
    'SL.IND.EMPL.ZS': 'emprego_industria_percent_emprego_total',
    'NV.IND.TOTL.ZS': 'valor_agregado_industrial_percent_pib'
}

data_inicio, data_fim = 1996, 2023

print("\nBaixando todos os indicadores de uma só vez (requisição única)...")
try:
    # Requisição única para todos os países e indicadores
    dados = wbdata.get_dataframe(indicadores_wdi, country=paises_dev_id)

    # Resetar índice e renomear
    dados = dados.reset_index()
    dados.rename(columns={'country': 'pais', 'date': 'ano'}, inplace=True)

    # Converter ano para inteiro
    dados['ano'] = pd.to_numeric(dados['ano'], errors='coerce')
    dados.dropna(subset=['ano'], inplace=True)

    # Filtrar período
    dados = dados[(dados['ano'] >= data_inicio) & (dados['ano'] <= data_fim)]

    # Salvar
    dados.to_csv('data/raw/wdi_emergentes_final.csv', index=False)
    dados.to_excel('data/raw/wdi_emergentes_final.xlsx', index=False)

    print(f"\n✅ Download concluído! {len(dados)} registros (pais-ano).")
    print("\nPrimeiras 10 linhas:")
    print(dados.head(10))

except Exception as e:
    print(f"❌ Erro na requisição única: {e}")
    print("Tentando país por país como fallback...")

    # Fallback: país por país (mas sem melt e concat desnecessários)
    dados_fallback = pd.DataFrame()
    for pais in paises_dev_id:
        try:
            df_pais = wbdata.get_dataframe(indicadores_wdi, country=pais)
            df_pais = df_pais.reset_index()
            df_pais.rename(columns={'country': 'pais', 'date': 'ano'}, inplace=True)
            df_pais['ano'] = pd.to_numeric(df_pais['ano'], errors='coerce')
            df_pais.dropna(subset=['ano'], inplace=True)
            dados_fallback = pd.concat([dados_fallback, df_pais], ignore_index=True)
        except Exception as e2:
            print(f"  Falhou para {pais}: {e2}")

    dados_fallback = dados_fallback[(dados_fallback['ano'] >= data_inicio) & (dados_fallback['ano'] <= data_fim)]
    dados_fallback.to_csv('data/raw/wdi_emergentes_final.csv', index=False)
    dados_fallback.to_excel('data/raw/wdi_emergentes_final.xlsx', index=False)
    print(f"\n✅ Fallback concluído! {len(dados_fallback)} registros.")

Obtendo países do Banco Mundial...
✅ Países em desenvolvimento (válidos): 131

Baixando todos os indicadores de uma só vez (requisição única)...

✅ Download concluído! 3668 registros (pais-ano).

Primeiras 10 linhas:
           pais   ano  pib_per_capita_ppc  \
2   Afghanistan  2023         1983.812620   
3   Afghanistan  2022         1981.710168   
4   Afghanistan  2021         2144.166570   
5   Afghanistan  2020         2769.685745   
6   Afghanistan  2019         2927.245144   
7   Afghanistan  2018         2902.392113   
8   Afghanistan  2017         2952.998916   
9   Afghanistan  2016         2958.785399   
10  Afghanistan  2015         2967.692067   
11  Afghanistan  2014         3017.942544   

    formacao_bruta_capital_fixo_percent_pib  \
2                                 15.243577   
3                                 16.668472   
4                                 12.986704   
5                                 11.455628   
6                                       NaN   
7    